# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasmeenmh90-beep/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:

# ML-05 — Section 1: Dataset connection
# Reuse the same March 2026 dataset as ML-04.

%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is missing from Colab Secrets."

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

fact_month = (
    f"read_parquet('{REL}/"
    f"fact_content_daily_performance/"
    f"month={MONTH}/*.parquet')"
)

summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {fact_month}
""").df()

print("ML-05 dataset connection successful")
display(summary)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ML-05 dataset connection successful


,total_rows,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# ML-05 — Section 1: Build the feature vector
# Prediction cutoff: end of March 15, 2026

import numpy as np

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- First-half GSC features
        SUM(CASE
            WHEN report_date <= DATE '2026-03-15'
            THEN COALESCE(gsc_impressions, 0)
            ELSE 0 END) AS imp_first_half,

        SUM(CASE
            WHEN report_date <= DATE '2026-03-15'
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0 END) AS clicks_first_half,

        AVG(CASE
            WHEN report_date <= DATE '2026-03-15'
            THEN gsc_avg_position
            END) AS pos_first_half,

        -- First-half GA4 features
        SUM(CASE
            WHEN report_date <= DATE '2026-03-15'
            THEN COALESCE(sessions_organic, 0)
            ELSE 0 END) AS organic_sessions_first_half,

        SUM(CASE
            WHEN report_date <= DATE '2026-03-15'
            THEN COALESCE(sessions_ai, 0)
            ELSE 0 END) AS ai_sessions_first_half,

        SUM(CASE
            WHEN report_date <= DATE '2026-03-15'
            THEN COALESCE(scroll_events, 0)
            ELSE 0 END) AS scroll_events_first_half,

        -- Availability indicators from first half only
        MAX(CASE
            WHEN report_date <= DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
            THEN 1 ELSE 0 END) AS has_gsc_history,

        MAX(CASE
            WHEN report_date <= DATE '2026-03-15'
                 AND ga4_data_available IS TRUE
            THEN 1 ELSE 0 END) AS has_ga4_history,

        -- Future period: LABEL CONSTRUCTION ONLY
        SUM(CASE
            WHEN report_date > DATE '2026-03-15'
            THEN COALESCE(gsc_impressions, 0)
            ELSE 0 END) AS imp_second_half

    FROM {fact_month}
    GROUP BY client_hash_id, content_hash_id
    HAVING imp_first_half >= 50
""").df()

# Target: at least a 20% decline
feature_frame["is_declining"] = (
    feature_frame["imp_second_half"]
    < 0.8 * feature_frame["imp_first_half"]
).astype(int)

# Engineered features
feature_frame["ctr_first_half"] = (
    feature_frame["clicks_first_half"]
    / feature_frame["imp_first_half"]
)

feature_frame["log_impressions"] = np.log1p(
    feature_frame["imp_first_half"]
)

# Preserve missingness explicitly
feature_frame["position_missing"] = (
    feature_frame["pos_first_half"].isna().astype(int)
)

feature_frame["pos_first_half"] = (
    feature_frame["pos_first_half"].fillna(
        feature_frame["pos_first_half"].median()
    )
)

# Keep identifiers, label ingredients, and future data
# OUT of the model's feature matrix.
feature_cols = [
    "imp_first_half",
    "clicks_first_half",
    "pos_first_half",
    "organic_sessions_first_half",
    "ai_sessions_first_half",
    "scroll_events_first_half",
    "has_gsc_history",
    "has_ga4_history",
    "ctr_first_half",
    "log_impressions",
    "position_missing",
]

X = feature_frame[feature_cols].copy()
y = feature_frame["is_declining"].copy()

print("Feature rows:", len(X))
print("Number of features:", X.shape[1])
print("Declining class rate:", round(y.mean(), 4))
print("Missing feature values:", int(X.isna().sum().sum()))
print("Duplicate client-content pairs:",
      int(feature_frame.duplicated(
          ["client_hash_id", "content_hash_id"]
      ).sum()))

display(X.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 92548
Number of features: 11
Declining class rate: 0.2864
Missing feature values: 0
Duplicate client-content pairs: 0


,imp_first_half,clicks_first_half,pos_first_half,organic_sessions_first_half,ai_sessions_first_half,scroll_events_first_half,has_gsc_history,has_ga4_history,ctr_first_half,log_impressions,position_missing
0,4173.0,6.0,6.327311,0.0,0.0,0.0,1,0,0.001438,8.336630,0
1,245.0,0.0,3.906852,0.0,0.0,0.0,1,0,0.000000,5.505332,0
2,3705.0,3.0,6.473735,0.0,0.0,0.0,1,0,0.000810,8.217708,0
3,2440.0,8.0,7.259861,0.0,0.0,0.0,1,0,0.003279,7.800163,0
4,240.0,1.0,3.860842,0.0,0.0,0.0,1,0,0.004167,5.484797,0


In [5]:



# ML-05 — Section 1: Feature quality checks

import pandas as pd

print("=== Feature availability ===")

availability = pd.DataFrame({
    "Indicator": [
        "GSC history",
        "GA4 history",
        "Missing GSC position"
    ],
    "Rows": [
        int(feature_frame["has_gsc_history"].sum()),
        int(feature_frame["has_ga4_history"].sum()),
        int(feature_frame["position_missing"].sum())
    ]
})

availability["Percentage"] = (
    100 * availability["Rows"] / len(feature_frame)
).round(2)

display(availability)

print("\n=== Feature variability ===")

quality = pd.DataFrame({
    "Feature": feature_cols,
    "Unique values": [
        X[col].nunique() for col in feature_cols
    ],
    "Zero percentage": [
        round(100 * (X[col] == 0).mean(), 2)
        for col in feature_cols
    ]
})

display(quality)

print("\n=== Label distribution ===")
display(y.value_counts().sort_index())

print("\n=== Feature validation ===")

assert len(X) == len(y)
assert X.shape[1] == 11
assert X.isna().sum().sum() == 0
assert not X.columns.isin([
    "imp_second_half",
    "is_declining",
    "client_hash_id",
    "content_hash_id"
]).any()

print("All structural checks passed.")


=== Feature availability ===


,Indicator,Rows,Percentage
0,GSC history,92548,100.00
1,GA4 history,32976,35.63
2,Missing GSC position,0,0.00



=== Feature variability ===


,Feature,Unique values,Zero percentage
0,imp_first_half,8874,0.00
1,clicks_first_half,241,46.92
2,pos_first_half,91675,0.00
3,organic_sessions_first_half,222,75.14
4,ai_sessions_first_half,20,98.24
5,scroll_events_first_half,103,79.03
6,has_gsc_history,1,0.00
7,has_ga4_history,2,64.37
8,ctr_first_half,18631,46.92
9,log_impressions,8874,0.00



=== Label distribution ===


,count
is_declining,
0,66039
1,26509



=== Feature validation ===
All structural checks passed.


In [6]:

# ML-05 — Section 1: Final feature vector

constant_features = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

X_final = X.drop(columns=constant_features).copy()

print("Removed constant features:", constant_features)
print("Final feature count:", X_final.shape[1])
print("Final feature rows:", X_final.shape[0])

assert X_final.shape[1] == 9
assert len(X_final) == len(y)
assert not X_final.isna().any().any()
assert X_final.select_dtypes(
    exclude="number"
).empty

print("\nFinal feature vector validated.")
display(X_final.head())


Removed constant features: ['has_gsc_history', 'position_missing']
Final feature count: 9
Final feature rows: 92548

Final feature vector validated.


,imp_first_half,clicks_first_half,pos_first_half,organic_sessions_first_half,ai_sessions_first_half,scroll_events_first_half,has_ga4_history,ctr_first_half,log_impressions
0,4173.0,6.0,6.327311,0.0,0.0,0.0,0,0.001438,8.336630
1,245.0,0.0,3.906852,0.0,0.0,0.0,0,0.000000,5.505332
2,3705.0,3.0,6.473735,0.0,0.0,0.0,0,0.000810,8.217708
3,2440.0,8.0,7.259861,0.0,0.0,0.0,0,0.003279,7.800163
4,240.0,1.0,3.860842,0.0,0.0,0.0,0,0.004167,5.484797


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*




### Feature definitions and availability

**Prediction task:** Predict whether a client-content pair will experience at least a 20% decline in GSC impressions during March 16–31, 2026, compared with March 1–15, 2026.

**Prediction cutoff:** End of March 15, 2026. All model features must be available by this cutoff. Second-half impressions are used only to construct the target label.

**Unit of analysis:** One row per `(client_hash_id, content_hash_id)` pair. The dataset contains 92,548 eligible rows, each with at least 50 first-half impressions.

| Feature | Meaning | Missing-value handling | Available when? |
|---|---|---|---|
| `imp_first_half` | Total GSC impressions during March 1–15 | Null daily counts replaced with zero during aggregation | After March 15 |
| `clicks_first_half` | Total GSC clicks during March 1–15 | Null daily counts replaced with zero | After March 15 |
| `pos_first_half` | Mean of reported daily GSC average positions | Median imputation implemented; no missing values observed | After March 15 |
| `organic_sessions_first_half` | Total organic sessions during March 1–15 | Null counts replaced with zero; GA4 availability must be considered | After March 15 |
| `ai_sessions_first_half` | Total AI-referred sessions during March 1–15 | Null counts replaced with zero; GA4 availability must be considered | After March 15 |
| `scroll_events_first_half` | Total recorded scroll events during March 1–15 | Null counts replaced with zero; GA4 availability must be considered | After March 15 |
| `has_ga4_history` | Indicates whether GA4 data was available during the first half | Binary 0/1 indicator | After March 15 |
| `ctr_first_half` | First-half clicks divided by first-half impressions | Denominator is positive because eligibility requires at least 50 impressions | After March 15 |
| `log_impressions` | Natural logarithm of `1 + imp_first_half` | Derived from first-half impressions | After March 15 |

### Categorical handling

The final feature vector contains eight numerical features and one binary availability indicator. No categorical encoding is required.

Client and content identifiers are retained for grouping and auditing but excluded from the modeling matrix to avoid learning identifier-specific patterns.

### Missingness and feature selection

The original 11 candidate features were reduced to 9 after removing two constant features:

- `has_gsc_history`: Equal to 1 for all eligible rows.
- `position_missing`: Equal to 0 for all eligible rows.

GA4 history was available for 32,976 of 92,548 eligible rows (35.63%). Consequently, zero-valued GA4 features cannot always be interpreted as genuine zero activity.

AI sessions were zero in 98.24% of rows, organic sessions in 75.14%, and scroll events in 79.03%. These features are retained as candidates, but their usefulness requires further validation.

### Prediction-time limitations

The features use records dated no later than March 15. However, their actual availability at prediction time depends on source-system reporting delays, which have not yet been verified.

The current implementation also treats missing daily counts as zero. This may affect interpretation and label reliability, particularly when GSC or GA4 data is unavailable.

These limitations will be examined in Section 3.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# ML-05 — Section 2: Feature documentation checks

import pandas as pd

feature_notes = pd.DataFrame({
    "feature": X_final.columns,
    "dtype": X_final.dtypes.astype(str).values,
    "missing_count": X_final.isna().sum().values,
    "unique_values": X_final.nunique().values,
    "available_by_cutoff": [True] * len(X_final.columns)
})

display(feature_notes)

# Confirm that future-period values and identifiers
# are not included in the feature matrix.
forbidden = {
    "imp_second_half",
    "is_declining",
    "client_hash_id",
    "content_hash_id"
}

assert forbidden.isdisjoint(X_final.columns)
assert X_final.shape == (92548, 9)

print("Section 2 structural checks passed.")
print("Reporting-lag verification remains outstanding.")


,feature,dtype,missing_count,unique_values,available_by_cutoff
0,imp_first_half,float64,0,8874,True
1,clicks_first_half,float64,0,241,True
2,pos_first_half,float64,0,91675,True
3,organic_sessions_first_half,float64,0,222,True
4,ai_sessions_first_half,float64,0,20,True
5,scroll_events_first_half,float64,0,103,True
6,has_ga4_history,int32,0,2,True
7,ctr_first_half,float64,0,18631,True
8,log_impressions,float64,0,8874,True


Section 2 structural checks passed.
Reporting-lag verification remains outstanding.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# ML-05 — Section 3: Leakage audit
# Test 1: Forbidden and label-derived columns

forbidden_columns = {
    "imp_second_half",
    "is_declining",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month"
}

detected = sorted(
    forbidden_columns.intersection(X_final.columns)
)

print("=== Target leakage audit ===")
print("Forbidden columns detected:", detected)

assert not detected, "Potential leakage detected!"

print("PASS: No forbidden columns in X_final.")

# Test 2: Verify label construction
expected_label = (
    feature_frame["imp_second_half"]
    < 0.8 * feature_frame["imp_first_half"]
).astype(int)

label_matches = (
    expected_label == feature_frame["is_declining"]
).all()

print("\n=== Label consistency ===")
print("Label calculation matches:", label_matches)

assert label_matches
print("PASS: Target calculation is consistent.")

# Test 3: Check feature matrix values
print("\n=== Feature matrix checks ===")

assert X_final.index.equals(y.index)
assert X_final.isna().sum().sum() == 0
assert len(X_final) == 92548

print("Feature rows:", len(X_final))
print("Feature columns:", X_final.shape[1])
print("PASS: Structural checks completed.")


=== Target leakage audit ===
Forbidden columns detected: []
PASS: No forbidden columns in X_final.

=== Label consistency ===
Label calculation matches: True
PASS: Target calculation is consistent.

=== Feature matrix checks ===
Feature rows: 92548
Feature columns: 9
PASS: Structural checks completed.


In [9]:

# ML-05 — Section 3: Second-half GSC coverage audit

coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT CASE
            WHEN report_date > DATE '2026-03-15'
            THEN report_date
        END) AS second_half_days,

        COUNT(DISTINCT CASE
            WHEN report_date > DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
            THEN report_date
        END) AS second_half_gsc_days

    FROM {fact_month}
    GROUP BY client_hash_id, content_hash_id
""").df()

coverage = feature_frame[
    ["client_hash_id", "content_hash_id", "is_declining"]
].merge(
    coverage,
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one"
)

coverage["second_half_days"] = (
    coverage["second_half_days"].fillna(0)
)

coverage["second_half_gsc_days"] = (
    coverage["second_half_gsc_days"].fillna(0)
)

print("=== Second-half GSC coverage ===")
print("Eligible items:", len(coverage))

print(
    "Items with zero second-half GSC days:",
    int((coverage["second_half_gsc_days"] == 0).sum())
)

print(
    "Items with fewer than 16 GSC days:",
    int((coverage["second_half_gsc_days"] < 16).sum())
)

print(
    "Items with all 16 GSC days:",
    int((coverage["second_half_gsc_days"] == 16).sum())
)

print("\n=== GSC day-count distribution ===")

display(
    coverage["second_half_gsc_days"]
    .value_counts()
    .sort_index()
    .rename_axis("GSC days")
    .reset_index(name="Content items")
)

print("\n=== Decline rate by coverage ===")

coverage["coverage_group"] = pd.cut(
    coverage["second_half_gsc_days"],
    bins=[-1, 0, 7, 15, 16],
    labels=["0 days", "1–7 days", "8–15 days", "16 days"]
)

display(
    coverage.groupby(
        "coverage_group", observed=True
    )["is_declining"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "decline_rate"})
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Second-half GSC coverage ===
Eligible items: 92548
Items with zero second-half GSC days: 301
Items with fewer than 16 GSC days: 12198
Items with all 16 GSC days: 80350

=== GSC day-count distribution ===


,GSC days,Content items
0,0,301
1,1,124
2,2,135
3,3,160
4,4,197
5,5,230
6,6,226
7,7,239
8,8,308
9,9,409



=== Decline rate by coverage ===


,count,decline_rate
coverage_group,,
0 days,301,1.000000
1–7 days,1311,0.974066
8–15 days,10586,0.605989
16 days,80350,0.230442


In [10]:

# ML-05 — Section 3: Label sensitivity analysis

# Join coverage information back to the feature dataset.
audit_frame = feature_frame.merge(
    coverage[
        ["client_hash_id",
         "content_hash_id",
         "second_half_gsc_days"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one"
)

# Complete observed second-half GSC coverage
full_coverage = audit_frame[
    audit_frame["second_half_gsc_days"] == 16
].copy()

print("=== Label sensitivity analysis ===")

print("Original eligible rows:", len(audit_frame))
print("Full-coverage rows:", len(full_coverage))
print(
    "Excluded from sensitivity subset:",
    len(audit_frame) - len(full_coverage)
)

print(
    "\nOriginal decline rate:",
    round(audit_frame["is_declining"].mean(), 4)
)

print(
    "Full-coverage decline rate:",
    round(full_coverage["is_declining"].mean(), 4)
)

# Compare class counts
comparison = pd.DataFrame({
    "Original": audit_frame["is_declining"]
        .value_counts().sort_index(),
    "Full coverage": full_coverage["is_declining"]
        .value_counts().sort_index()
}).fillna(0).astype(int)

display(comparison)

# Structural validation
assert len(full_coverage) == 80350
assert full_coverage["second_half_gsc_days"].eq(16).all()

print("\nSensitivity analysis completed.")
print(
    "Note: Full coverage is a retrospective "
    "analysis subset, not a prediction feature."
)


=== Label sensitivity analysis ===
Original eligible rows: 92548
Full-coverage rows: 80350
Excluded from sensitivity subset: 12198

Original decline rate: 0.2864
Full-coverage decline rate: 0.2304


,Original,Full coverage
is_declining,,
0,66039,61834
1,26509,18516



Sensitivity analysis completed.
Note: Full coverage is a retrospective analysis subset, not a prediction feature.


In [11]:

# ML-05 — Section 3: Product flag and availability audit

flags = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        MAX(CASE
            WHEN report_date <= DATE '2026-03-15'
                 AND client_has_ga4 IS TRUE
            THEN 1 ELSE 0 END) AS client_has_ga4_flag,

        MAX(CASE
            WHEN report_date <= DATE '2026-03-15'
                 AND ga4_data_available IS TRUE
            THEN 1 ELSE 0 END) AS ga4_available_flag

    FROM {fact_month}
    GROUP BY client_hash_id, content_hash_id
""").df()

flag_audit = feature_frame[
    ["client_hash_id", "content_hash_id",
     "has_ga4_history"]
].merge(
    flags,
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one"
)

print("=== Product flag audit ===")

print("Eligible records:", len(flag_audit))

print(
    "Client GA4 enabled:",
    int(flag_audit["client_has_ga4_flag"].sum())
)

print(
    "GA4 data available:",
    int(flag_audit["ga4_available_flag"].sum())
)

print("\n=== Flag comparison ===")

display(pd.crosstab(
    flag_audit["client_has_ga4_flag"],
    flag_audit["ga4_available_flag"],
    rownames=["Client GA4 enabled"],
    colnames=["GA4 data available"]
))

print("\n=== Consistency check ===")

assert (
    flag_audit["has_ga4_history"]
    == flag_audit["ga4_available_flag"]
).all()

print("PASS: Feature availability flag matches source.")
print("Reporting timestamps still require verification.")


=== Product flag audit ===
Eligible records: 92548
Client GA4 enabled: 49273
GA4 data available: 32976

=== Flag comparison ===


GA4 data available,0,1
Client GA4 enabled,,
0,43275,0
1,16297,32976



=== Consistency check ===
PASS: Feature availability flag matches source.
Reporting timestamps still require verification.


### Leakage and privacy investigation

**Prediction cutoff:** March 15, 2026.

The final feature vector contains nine candidate features derived from the first-half observation window. The target label is calculated from second-half GSC impressions.

**1. Target-derived feature leakage**

The audit confirmed that `imp_second_half`, `is_declining`, client identifiers, content identifiers, report dates and month identifiers are absent from the final modeling matrix.

The label consistency test passed, and the final matrix contains 92,548 records and nine features.

**2. Future-window contamination and label reliability**

Second-half GSC coverage was examined for all eligible records.

- 80,350 records had 16 observed GSC days.
- 12,198 records had fewer than 16 observed GSC days.
- 301 records had no observed second-half GSC days.

The original decline rate was 28.64%, compared with 23.04% among records with 16 observed GSC days.

All 301 records with zero observed GSC days were labeled as declining. This demonstrates a potential label-quality issue because the current target construction treats unavailable impression counts as zero.

However, incomplete observation coverage does not necessarily indicate missing data. Genuine zero-traffic periods may also produce fewer observations.

Second-half coverage is retained only for retrospective auditing and must not be included as a prediction feature.

**3. Product flag and data availability checks**

The first-half GA4 audit identified:

- 43,275 records with GA4 not enabled.
- 16,297 records with GA4 enabled but no available measurements.
- 32,976 records with GA4 enabled and available measurements.

The engineered `has_ga4_history` feature exactly matched the first-half `ga4_data_available` indicator.

The `client_has_ga4` product flag was not added to the final feature vector. Product enablement and actual measurement availability are not equivalent.

**4. Privacy and identifier handling**

The modeling matrix excludes `client_hash_id` and `content_hash_id`. These identifiers are used only for grouping, joining and auditing.

No raw client names, private URLs or search queries were introduced into the feature vector. Hashed identifiers are still potentially sensitive and should not be treated as automatically anonymous.

**5. Remaining limitations**

The audits establish that explicitly forbidden columns are absent and that the tested feature definitions use first-half records.

They do not establish that source measurements were actually available by the prediction cutoff. Reporting delays, retrospective corrections and historical availability of product flags remain unverified.

The observed association between GSC coverage and decline labels warrants further investigation before production use.

**Conclusion:** The final feature vector passes the implemented structural leakage checks. Potential label bias and prediction-time data availability remain unresolved risks.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded features and reasons

| Excluded field | Reason |
|---|---|
| `imp_second_half` | Future-period impressions directly used to calculate the target; target leakage |
| `is_declining` | Prediction target; cannot be used as an input |
| `second_half_gsc_days` | Future-period coverage information; retrospective audit only |
| `client_hash_id` | Identifier excluded from modeling to reduce memorization and privacy risks |
| `content_hash_id` | Identifier excluded from modeling to reduce memorization and privacy risks |
| `report_date` | Used for filtering and defining observation windows, not directly modeled |
| `month` | Partition and filtering field, not a modeling feature |
| `has_gsc_history` | Constant at 1 across all eligible records |
| `position_missing` | Constant at 0 across all eligible records |
| `client_has_ga4` | Product enablement flag; excluded because enablement does not guarantee available measurements |
| `gsc_sum_position` | Not selected for the current feature vector; requires separate validation of its aggregation and predictive usefulness |
| Second-half clicks, sessions and engagement metrics | Unavailable at the prediction cutoff; potential future-data leakage |

The final modeling matrix retains nine candidate features. Exclusions are based on leakage prevention, privacy considerations, lack of variation, or the chosen feature-engineering scope.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-05 — Section 4: Final exclusion audit

# Features that must never enter the model
forbidden_features = {
    "imp_second_half",
    "is_declining",
    "second_half_gsc_days",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "client_has_ga4",
    "gsc_sum_position",
}

# Constant features removed during engineering
constant_features = {
    "has_gsc_history",
    "position_missing",
}

# Check final feature matrix
final_columns = set(X_final.columns)

forbidden_found = sorted(final_columns & forbidden_features)
constants_found = sorted(final_columns & constant_features)

print("=== FINAL FEATURE EXCLUSION AUDIT ===")
print("Feature rows:", len(X_final))
print("Feature columns:", len(X_final.columns))
print("Forbidden features found:", forbidden_found)
print("Constant features found:", constants_found)
print("Missing values:", int(X_final.isna().sum().sum()))

assert len(X_final) == 92548, "Unexpected row count"
assert X_final.shape[1] == 9, "Unexpected feature count"
assert not forbidden_found, "Forbidden features detected"
assert not constants_found, "Constant features detected"
assert X_final.isna().sum().sum() == 0, "Missing values detected"

print("\nPASS: Final feature exclusion audit completed.")

=== FINAL FEATURE EXCLUSION AUDIT ===
Feature rows: 92548
Feature columns: 9
Forbidden features found: []
Constant features found: []
Missing values: 0

PASS: Final feature exclusion audit completed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.